In [ ]:
# Set up
from google.colab import drive
drive.mount('/content/drive')
!pip install mne --quiet
import os
import mne
import glob
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d
from scipy.fft import fft, ifft
from tqdm import tqdm
import scipy.io.wavfile as wav
from mne.time_frequency import tfr_array_morlet

In [ ]:
# Morlet Wavelet Analysis

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
from scipy.ndimage import gaussian_filter1d
from mne.time_frequency import tfr_array_morlet

# ---------------------
# PARAMETERS
# ---------------------
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli/'
wav_files = sorted([
    f for f in glob.glob(os.path.join(root_dir, '*.wav'))
    if f.endswith('_N.wav') or f.endswith('_E.wav')
])

# Improved parameters:
freqs = np.logspace(np.log10(80), np.log10(5000), 80)  # log-spaced frequencies
n_cycles = 5                                            # more stable envelope
smooth_ms = 10                                          # slightly more smoothing

# ---------------------
# PROCESSING LOOP
# ---------------------
for wav_path in wav_files:
    fs, data = wav.read(wav_path)
    if data.ndim > 1:
        data = np.sqrt(np.mean(data.astype(np.float32)**2, axis=1))
    else:
        data = data.astype(np.float32)

    data = data / np.max(np.abs(data))  # normalize to -1...1
    t = np.arange(len(data)) / fs

    # Reshape for MNE (1 "epoch", 1 "channel", n_times)
    data_reshaped = data[np.newaxis, np.newaxis, :]

    # Compute Morlet-based TFR amplitude
    power = tfr_array_morlet(data_reshaped, sfreq=fs, freqs=freqs,
                              n_cycles=n_cycles, output='power', zero_mean=True)

    # Compute broadband envelope: average across frequencies
    envelope_wavelet_avg = np.mean(np.sqrt(power[0, 0, :, :]), axis=0)

    # Smooth envelope
    sigma = (smooth_ms / 1000.0) * fs
    envelope_smooth = gaussian_filter1d(envelope_wavelet_avg, sigma)

    # Compute first derivative
    dt = 1.0 / fs
    envelope_derivative = np.gradient(envelope_smooth, dt)

    # ---- PLOTTING ----

    plt.figure(figsize=(12, 5))
    plt.plot(t, data, linewidth=0.5, label='Waveform', color='steelblue')
    plt.plot(t, envelope_smooth, linewidth=2, label='Wavelet Envelope', color='orange')
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.title('Wavelet-based Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(t, envelope_derivative, color='green', label='Envelope Derivative')
    plt.xlabel('Time [s]')
    plt.ylabel('dEnvelope/dt')
    plt.title('First Derivative of Wavelet Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()
